In [57]:
# IMPORTS

# pip install scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn import metrics
from ydata_profiling import ProfileReport

In [58]:
df = pd.read_csv('lego_sets.csv')
df.head()

,ages,list_price,num_reviews,piece_count,play_star_rating,prod_desc,prod_id,prod_long_desc,review_difficulty,set_name,star_rating,theme_name,val_star_rating,country
0,6-12,29.99,2.0,277.0,4.0,Catapult into action and take back the eggs fr...,75823.0,Use the staircase catapult to launch Red into ...,Average,Bird Island Egg Heist,4.5,Angry Birds™,4.0,US
1,6-12,19.99,2.0,168.0,4.0,Launch a flying attack and rescue the eggs fro...,75822.0,Pilot Pig has taken off from Bird Island with ...,Easy,Piggy Plane Attack,5.0,Angry Birds™,4.0,US
2,6-12,12.99,11.0,74.0,4.3,Chase the piggy with lightning-fast Chuck and ...,75821.0,Pitch speedy bird Chuck against the Piggy Car....,Easy,Piggy Car Escape,4.3,Angry Birds™,4.1,US
3,12+,99.99,23.0,1032.0,3.6,Explore the architecture of the United States ...,21030.0,Discover the architectural secrets of the icon...,Average,United States Capitol Building,4.6,Architecture,4.3,US
4,12+,79.99,14.0,744.0,3.2,Recreate the Solomon R. Guggenheim Museum® wit...,21035.0,Discover the architectural secrets of Frank Ll...,Challenging,Solomon R. Guggenheim Museum®,4.6,Architecture,4.1,US


### step 0:
creating new interesting columns and fixing current ones


In [59]:
df['country'].unique()

array(['US', 'AU', 'AT', 'BE', 'CA', 'CH', 'CZ', 'DE', 'DN', 'ES', 'FI',
       'FR', 'GB', 'IE', 'IT', 'LU', 'NO', 'NL', 'NZ', 'PL', 'PT'],
      dtype=object)

In [60]:
# This code makes the column 'continent' instead of country, which can be then turned into three separate columns.

# country_to_continent = {
#     'US': 'North America',
#     'CA': 'North America',
#     'AU': 'Oceania',
#     'NZ': 'Oceania',
#     'AT': 'Europe',
#     'BE': 'Europe',
#     'CH': 'Europe',
#     'CZ': 'Europe',
#     'DE': 'Europe',
#     'DN': 'Europe',   
#     'ES': 'Europe',
#     'FI': 'Europe',
#     'FR': 'Europe',
#     'GB': 'Europe',
#     'IE': 'Europe',
#     'IT': 'Europe',
#     'LU': 'Europe',
#     'NO': 'Europe',
#     'NL': 'Europe',
#     'PL': 'Europe',
#     'PT': 'Europe',
# }

# df['continent'] = df['country'].map(country_to_continent)
# df = df.drop('country', axis=1)

In [61]:
# Creating a new column 'price_per_brick' for more fun comparisons
# Some legosets might be more expensive even with less bricks than other sets
# Only for analytic purposes, doesn't work in ML

df['price_per_brick'] = df['list_price'] / df['piece_count']
df['price_per_brick'] = df['price_per_brick'].round(2)
df['price_per_brick']


0        0.11
1        0.12
2        0.18
3        0.10
4        0.11
         ... 
12256    0.11
12257    0.11
12258    0.10
12259    0.25
12260    0.11
Name: price_per_brick, Length: 12261, dtype: float64

In [62]:
df['ages'].value_counts()
# Separate this column to two new columns - minAge and maxAge

ages
6-12     1476
7-14     1421
8-14     1180
4-7       957
5-12      911
10+       870
2-5       840
7-12      723
9-14      624
16+       420
8-12      350
4-99      311
12+       298
6-14      233
8+        226
1½-3      213
14+       212
10-21     184
6+        148
10-16     148
1½-5      113
9-16       92
5+         71
11-16      66
9-12       46
12-16      42
4+         21
9+         21
5-8        21
10-14      21
7+          2
Name: count, dtype: int64

In [63]:
# This code is 98% claude
# The primary function is to separate the 'ages' column into two different columns - min and max ages

import re

def parse_age(age_str):
    if pd.isna(age_str):
        return pd.Series([None, None])
    
    age_str = str(age_str).strip()
    age_str = age_str.replace('½', '.5')  # handle fractions like 1½ -> 1.5
    
    # Case: "99+" or "12+"
    if '+' in age_str:
        match = re.search(r'[\d.]+', age_str)
        num = float(match.group()) if match else None
        return pd.Series([num, None])
    
    # Case: "12-16" (range)
    if '-' in age_str:
        parts = age_str.split('-')
        try:
            return pd.Series([float(parts[0].strip()), float(parts[1].strip())])
        except (ValueError, IndexError):
            return pd.Series([None, None])
    
    # Case: single number like "2"
    match = re.fullmatch(r'[\d.]+', age_str)
    if match:
        num = float(age_str)
        return pd.Series([num, num])
    
    return pd.Series([None, None])

df[['min_age', 'max_age']] = df['ages'].apply(parse_age)

df['max_age'] = df['max_age'].fillna(99) # To fill the na, assuming the lego sets without max age have the max age of 99
df = df.drop('ages', axis=1)

In [64]:
df.columns

Index(['list_price', 'num_reviews', 'piece_count', 'play_star_rating',
       'prod_desc', 'prod_id', 'prod_long_desc', 'review_difficulty',
       'set_name', 'star_rating', 'theme_name', 'val_star_rating', 'country',
       'price_per_brick', 'min_age', 'max_age'],
      dtype='object')

### step 1a:

In [65]:
df.dtypes
# They seem to be correct

list_price           float64
num_reviews          float64
piece_count          float64
play_star_rating     float64
prod_desc             object
prod_id              float64
prod_long_desc        object
review_difficulty     object
set_name              object
star_rating          float64
theme_name            object
val_star_rating      float64
country               object
price_per_brick      float64
min_age              float64
max_age              float64
dtype: object

### step 1b:

In [66]:
# # Making the ydata profile

# report = ProfileReport(df,
#     title='Lego sets / Advanced Data Analytics, 2026'
# )
# report.to_file("lego_sets_regression_report.html")

In [67]:
len(df)

12261

In [68]:
df.columns

Index(['list_price', 'num_reviews', 'piece_count', 'play_star_rating',
       'prod_desc', 'prod_id', 'prod_long_desc', 'review_difficulty',
       'set_name', 'star_rating', 'theme_name', 'val_star_rating', 'country',
       'price_per_brick', 'min_age', 'max_age'],
      dtype='object')

In [69]:
df.head()

,list_price,num_reviews,piece_count,play_star_rating,prod_desc,prod_id,prod_long_desc,review_difficulty,set_name,star_rating,theme_name,val_star_rating,country,price_per_brick,min_age,max_age
0,29.99,2.0,277.0,4.0,Catapult into action and take back the eggs fr...,75823.0,Use the staircase catapult to launch Red into ...,Average,Bird Island Egg Heist,4.5,Angry Birds™,4.0,US,0.11,6.0,12.0
1,19.99,2.0,168.0,4.0,Launch a flying attack and rescue the eggs fro...,75822.0,Pilot Pig has taken off from Bird Island with ...,Easy,Piggy Plane Attack,5.0,Angry Birds™,4.0,US,0.12,6.0,12.0
2,12.99,11.0,74.0,4.3,Chase the piggy with lightning-fast Chuck and ...,75821.0,Pitch speedy bird Chuck against the Piggy Car....,Easy,Piggy Car Escape,4.3,Angry Birds™,4.1,US,0.18,6.0,12.0
3,99.99,23.0,1032.0,3.6,Explore the architecture of the United States ...,21030.0,Discover the architectural secrets of the icon...,Average,United States Capitol Building,4.6,Architecture,4.3,US,0.10,12.0,99.0
4,79.99,14.0,744.0,3.2,Recreate the Solomon R. Guggenheim Museum® wit...,21035.0,Discover the architectural secrets of Frank Ll...,Challenging,Solomon R. Guggenheim Museum®,4.6,Architecture,4.1,US,0.11,12.0,99.0


In [74]:
# dropping unnessary columns, most of these columns are just set names or descriptions of the sets
droppables = ['prod_desc', 'prod_long_desc', 'theme_name', 'set_name']
df = df.drop(droppables, axis=1)

### Step 1c:

In [70]:
df.isna().sum()

list_price              0
num_reviews          1620
piece_count             0
play_star_rating     1775
prod_desc             377
prod_id                 0
prod_long_desc          0
review_difficulty    2055
set_name                0
star_rating          1620
theme_name              3
val_star_rating      1795
country                 0
price_per_brick         0
min_age                 0
max_age                 0
dtype: int64

In [71]:
# No rows with all rows na
df[df.isna().all(axis=1)]

,list_price,num_reviews,piece_count,play_star_rating,prod_desc,prod_id,prod_long_desc,review_difficulty,set_name,star_rating,theme_name,val_star_rating,country,price_per_brick,min_age,max_age


In [72]:
df[df.isna().any(axis=1)]

,list_price,num_reviews,piece_count,play_star_rating,prod_desc,prod_id,prod_long_desc,review_difficulty,set_name,star_rating,theme_name,val_star_rating,country,price_per_brick,min_age,max_age
22,9.9900,1.0,136.0,NaN,Master the skills of a green-skinned assassin ...,41607.0,This Gamora LEGO® BrickHeadz construction char...,NaN,Gamora,5.0,BrickHeadz,NaN,US,0.07,10.0,99.0
32,19.9900,NaN,209.0,NaN,Join the Justice League™ with LEGO® BrickHeadz...,41610.0,These LEGO® BrickHeadz™ 41610 Tactical Batman™...,NaN,Tactical Batman™ & Superman™,NaN,BrickHeadz,NaN,US,0.10,10.0,99.0
48,49.9900,NaN,387.0,NaN,Keep the crooks from escaping with the gold bars!,60175.0,Pick up your badge and join the LEGO® City Mou...,NaN,Mountain River Heist,NaN,City,NaN,US,0.13,5.0,12.0
55,99.9900,1.0,883.0,5.0,Head to the LEGO® City mine and dig up some gold!,60188.0,Grab your hard hat and head out to the LEGO® C...,NaN,Mining Experts Site,5.0,City,5.0,US,0.11,7.0,12.0
69,39.9900,NaN,297.0,NaN,Race after the crook and avoid the mountain ro...,60172.0,Pick up your badge and join the LEGO® City Mou...,NaN,Dirt Road Pursuit,NaN,City,NaN,US,0.13,5.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12136,36.5878,NaN,104.0,NaN,Own a true Sith Lord with Darth Maul!,75537.0,Bolster your LEGO® Star Wars collection with t...,NaN,Darth Maul™,NaN,Star Wars™,NaN,PT,0.35,8.0,14.0
12137,36.5878,NaN,101.0,NaN,Battle the bad guys with Han Solo!,75535.0,Play out exciting adventures with this highly ...,NaN,Han Solo™,NaN,Star Wars™,NaN,PT,0.36,8.0,14.0
12196,243.9878,NaN,1967.0,NaN,Collect the ultimate long-range Rebel starfigh...,75181.0,Own part of Star Wars history with the Y-Wing ...,NaN,Y-Wing Starfighter™,NaN,Star Wars™,NaN,PT,0.12,14.0,99.0
12214,24.3878,1.0,135.0,5.0,Get ready for high-speed action and impacts wi...,42072.0,Experience the amazing power and acceleration ...,NaN,WHACK!,5.0,Technic,5.0,PT,0.18,7.0,14.0


In [76]:
df_duplicates = df[df.duplicated()]

In [77]:
df_duplicates

,list_price,num_reviews,piece_count,play_star_rating,prod_id,review_difficulty,star_rating,val_star_rating,country,price_per_brick,min_age,max_age
195,9.9900,5.0,143.0,4.6,41599.0,Easy,5.0,5.0,US,0.07,10.0,99.0
196,9.9900,3.0,122.0,2.7,41598.0,Very Easy,4.0,3.0,US,0.08,10.0,99.0
197,9.9900,1.0,135.0,1.0,41600.0,Very Easy,4.0,3.0,US,0.07,10.0,99.0
198,9.9900,2.0,108.0,3.5,41601.0,Easy,5.0,4.0,US,0.09,10.0,99.0
199,19.9900,NaN,209.0,NaN,41610.0,NaN,NaN,NaN,US,0.10,10.0,99.0
...,...,...,...,...,...,...,...,...,...,...,...,...
12256,36.5878,6.0,341.0,4.4,70609.0,Easy,4.3,4.2,PT,0.11,7.0,14.0
12257,24.3878,8.0,217.0,4.1,70629.0,Easy,3.6,4.1,PT,0.11,7.0,14.0
12258,24.3878,18.0,233.0,4.6,70607.0,Easy,4.6,4.5,PT,0.10,7.0,14.0
12259,12.1878,1.0,48.0,5.0,70628.0,Very Easy,5.0,5.0,PT,0.25,6.0,14.0


In [18]:
df = df.dropna() # Drops 1/10th of the data so not too bad
len(df)

10165

In [20]:
df.duplicated().sum()

np.int64(1074)

In [21]:
df.drop_duplicates()

,list_price,num_reviews,piece_count,play_star_rating,review_difficulty,star_rating,val_star_rating,country,price_per_brick,min_age,max_age
0,29.9900,2.0,277.0,4.0,Average,4.5,4.0,US,0.11,6.0,12.0
1,19.9900,2.0,168.0,4.0,Easy,5.0,4.0,US,0.12,6.0,12.0
2,12.9900,11.0,74.0,4.3,Easy,4.3,4.1,US,0.18,6.0,12.0
3,99.9900,23.0,1032.0,3.6,Average,4.6,4.3,US,0.10,12.0,99.0
4,79.9900,14.0,744.0,3.2,Challenging,4.6,4.1,US,0.11,12.0,99.0
...,...,...,...,...,...,...,...,...,...,...,...
12234,42.6878,18.0,254.0,4.2,Easy,4.2,4.4,PT,0.17,7.0,14.0
12236,30.4878,8.0,204.0,4.1,Average,4.4,4.1,PT,0.15,7.0,14.0
12237,30.4878,19.0,201.0,4.1,Easy,4.6,4.4,PT,0.15,7.0,14.0
12238,18.2878,7.0,141.0,4.0,Easy,4.1,4.4,PT,0.13,6.0,12.0


In [26]:
# df.isna().sum() # Lots of NAs, trying KNNimputer for now

# from sklearn.impute import KNNImputer

# imputer = KNNImputer(n_neighbors=5)

# df = pd.DataFrame(
#     imputer.fit_transform(df),
#     columns=df.columns,
#     index=df.index
# )

# int_cols = ['review_difficulty'] 
# df[int_cols] = df[int_cols].round().astype(int)

In [ ]:
# # Mapping the review_difficulty column
# review_mapper = {'Very Easy': 0, 'Easy': 1, 'Average': 2, 'Challenging': 3, 'Very Challenging': 4}
# df['review_difficulty'] = df['review_difficulty'].map(review_mapper)
